In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, get_dataset_config_names
from collections import Counter
import random

# =================================================================================================
# 🌳🔥 데이터셋: FireRisk (화재 위험 지역 분류) 🔥🌳
# 📋 개요: 이 데이터셋은 원격 감지(Remote Sensing) 이미지를 사용하여 산불 위험도를 분류하는 태스크를 수행합니다.
# 💡 난이도: 초급 (탐색 및 데이터 분포 분석)
# 🎯 목표: 우리가 위성 이미지 데이터를 받는 '전문 분석가'가 되어, 어떤 위험 지역이 가장 많은지 분포를 분석하고, 특정 위험 등급만 따로 골라내는(필터링) 과정을 시뮬레이션합니다.
# =================================================================================================

# --- 설정 값 정의 ---
DATASET_NAME = "blanchon/FireRisk"
SAMPLE_COUNT = 50 # 너무 크지 않게, 상위 50개 샘플만 분석합니다.

print("👋 안녕하세요! 위성 이미지 분석가가 되어볼 준비가 되셨나요? 파이썬 코딩으로 지구의 위험 요소를 찾아보아요!")

# 1. 데이터셋 로드 전략 설정 및 스트리밍 여부 확인
print("\n[STEP 1] 🌍 데이터셋 로드 준비: 방대한 위성 이미지 데이터를 불러올 준비를 합니다...")
dataset = None
try:
    # 🌟 전략 1: 스트리밍 방식으로 로드 시도 (메모리 효율성을 위해 최우선)
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 스트리밍 모드(Streaming)로 데이터셋 연결에 성공했습니다. 메모리 걱정 없이 분석을 시작해요.")

except Exception as e:
    # 🚨 실패 시: 스트리밍이 어렵다면, 작은 분할(split)만 다운로드하여 진행
    print(f"⚠️ 스트리밍 로드 실패: {e}. 일반 모드로 전환하여 작은 샘플만 다운로드 진행합니다.")
    try:
        # 이 경우, 일반 Dataset 객체가 됩니다.
        dataset = load_dataset(DATASET_NAME, split='train', streaming=False)
        print("✅ 일반 모드로 데이터셋 로드 완료! 🚀")
    except Exception as e_fallback:
        print(f"🚨 치명적 오류: 데이터셋 로드에 실패했습니다. {e_fallback}")
        exit()


# 2. 데이터 샘플링 (Constraint Adherence)
print(f"\n[STEP 2] ✨ 샘플 추출: 전체 데이터셋에서 무작위로 {SAMPLE_COUNT}개의 샘플만 추출하여 빠르게 분석해볼게요.")

# 데이터를 list()로 변환하는 것이 스트리밍/일반 모드 모두에서 안전한 패턴입니다.
# (Constraint 9 & 16 준수)
sampled_dataset = list(dataset.take(SAMPLE_COUNT))

if not sampled_dataset:
    print("❌ 추출된 샘플이 없습니다. 데이터셋 로드를 다시 확인해주세요.")
    exit()

# 3. 라벨 매핑 및 분포 분석
print("\n[STEP 3] 📊 위험 등급 분석: 추출된 샘플들의 '위험도' 라벨을 계산해 봅시다.")

# 라벨 이름 매핑 딕셔너리 생성 (사람이 읽기 쉬운 이름으로 변환)
label_map = {
    '0': 'High Risk (High)',
    '1': 'Low Risk (Low)',
    '2': 'Moderate Risk (Moderate)',
    '3': 'Non-Burnable (Safe)',
    '4': 'Very High Risk (Very High)',
    '5': 'Very Low Risk (Very Low)',
    '6': 'Water (Water)'
}

label_labels = []
for sample in sampled_dataset:
    # 라벨은 문자열 형태의 클래스 레이블이므로, 딕셔너리 조회 방식으로 접근합니다.
    label_str = sample['label']
    label_name = label_map.get(label_str, f"Unknown Label ({label_str})")
    label_labels.append(label_name)

# Counter를 사용하여 각 라벨이 몇 번 나왔는지 카운트합니다.
label_counts = Counter(label_labels)

print("\n==========================================================")
print("🔥 🏆 종합 위험도 분포 분석 결과 (Top {SAMPLE_COUNT} samples)")
print("==========================================================")
for label, count in label_counts.items():
    print(f"  • {label:<20}: {count} 개")

# 가장 높은 위험도(Very High)와 가장 안전한(Non-Burnable)의 개수를 바로 비교해 봅니다.
very_high_count = label_counts.get('Very High Risk (Very High)', 0)
non_burnable_count = label_counts.get('Non-Burnable (Safe)', 0)

if very_high_count > 0 and non_burnable_count > 0:
    print(f"\n🔍 [분석 요약]: '매우 위험' 지역은 {very_high_count}개, '불연성' 지역은 {non_burnable_count}개로 나타났습니다.")
    if very_high_count > non_burnable_count:
        print("   -> 현재 샘플에서는 위험 구역이 더 많네요! 주의가 필요해요. 🧐")
    else:
        print("   -> 비교적 안정적인 지역이 많습니다. (만세! 🎉)")


# 4. 시각화: 라벨 분포 막대 그래프 생성
print("\n[STEP 4] 📈 시각화: 분포를 막대 그래프로 그려보겠습니다. 데이터가 눈으로도 보일수록 재미있죠!")

# matplotlib를 사용하여 시각화합니다. (Constraint 19 준수)
labels = list(label_counts.keys())
counts = list(label_counts.values())

plt.figure(figsize=(12, 6))
plt.bar(labels, counts, color='#d9534f') # 빨간 계열 색상을 사용해 위험 느낌 부여
plt.xlabel("Fire Risk Category")
plt.ylabel("Frequency (Count)")
plt.title(f"Distribution of Fire Risk Labels (Sample Size: {SAMPLE_COUNT})")
plt.xticks(rotation=45, ha='right') # x축 라벨이 길어지므로 45도 회전
plt.tight_layout()
plt.show()

# 5. 시뮬레이션: 특정 조건을 가진 데이터 필터링 (고급 분석가 모드)
print("\n[STEP 5] 🎯 필터링 시뮬레이션: '매우 위험(Very High)' 지역만 추려내는 필터를 작동시켜 봅시다.")

# 'Very High Risk' 라벨을 가진 샘플의 개수를 세어봅니다.
filtered_labels = [label for label in label_labels if label == 'Very High Risk (Very High)']
filtered_count = len(filtered_labels)

print(f"   >>> 성공! {filtered_count}개의 '매우 위험' 지역 샘플을 성공적으로 분리했습니다.")

if filtered_count > 0:
    print("   (이 샘플들은 집중 감시가 필요한 지역입니다. 위성 분석 전문가는 이 데이터를 다음 단계로 넘깁니다!)")
else:
    print("   이번 샘플에서는 '매우 위험'한 지역이 발견되지 않았습니다. 안심해도 좋아요! 😌")

print("\n✅ 모든 실습이 끝났습니다! 데이터의 구조를 이해하고, 분포를 분석하며, 원하는 조건으로 데이터를 추출하는 과정을 완벽하게 마쳤어요. 수고하셨습니다, 분석가님!")